In [1]:
%pip install git+https://github.com/timaeus-research/devinterp.git
%pip install seaborn torchvision pickleshare wandb plotly einops scikit-learn
%pip install git+https://github.com/MFreidank/pysgmcmc@pytorch
!git clone https://github.com/ucla-vision/entropy-sgd.git
%cd entropy-sgd
from python.optim import EntropySGD
%cd ..

Defaulting to user installation because normal site-packages is not writeable
  Cloning https://github.com/timaeus-research/devinterp.git to /gpfs/scratch1/nodespecific/gcn54/bshaffrey.12862092/pip-req-build-_7nq28zu
  Running command git clone --filter=blob:none --quiet https://github.com/timaeus-research/devinterp.git /gpfs/scratch1/nodespecific/gcn54/bshaffrey.12862092/pip-req-build-_7nq28zu
  Resolved https://github.com/timaeus-research/devinterp.git to commit b9b007a5b13641dfdabbb172b740b12e45cb6359
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached matplotlib-3.10.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (8.6 MB)
  Using cached cloudpickle-3.1.1-py3-none-any.whl (20 kB)

[notice] A new release of pip is available: 23.1.2 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Defaulting to use

In [2]:
import os
import torch
import numpy as np
import random
import torch.nn as nn
import seaborn as sns
from scipy import stats
from typing import Tuple
import torch.nn.functional as F
from matplotlib import pyplot as plt
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader


from toy_model import train_toy_model, load_model, get_dataloader, toy_model_loss
from sae import *
import pysgmcmc.optimizers.sghmc as sghmc

PRIMARY, SECONDARY, TERTIARY, QUATERNARY, QUINARY, SENARY = sns.color_palette("muted")[:6]
PRIMARY_LIGHT, SECONDARY_LIGHT, TERTIARY_LIGHT, QUATERNARY_LIGHT, QUINARY_LIGHT, SENARY_LIGHT = sns.color_palette(
    "pastel"
)[:6]

def set_seed(seed):
    # PyTorch seeds
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if using multiple GPUs
    
    # Python's random seed
    random.seed(seed)
    
    # NumPy seed
    np.random.seed(seed)
    
    # Make CUDA operations deterministic
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [3]:
seed = 42
set_seed(seed)

n_hidden = 2
n_features = n_hidden * 3
n_dictionary = 8 * n_features
l1_lambda = 3.0
n_sparsities = 1
epochs = 100
log_freq = epochs // 100 if epochs >= 100 else 1
n_samples = 10000
batch_size = 20
limit = True
temperature = 0.0
criterion = nn.MSELoss()
save_path = f"results/"
os.makedirs(save_path, exist_ok=True)
'''
results = train_sweep_of_sae_on_toy_model(
    n_hidden, 
    n_features, 
    n_dictionary, 
    l1_lambda, 
    save_path, 
    n_sparsities=n_sparsities, 
    epochs=epochs,
    log_freq=log_freq,
    n_samples=n_samples,
    batch_size=batch_size,
    limit=limit,
    temperature=temperature,
    criterion=criterion
)
torch.save(results, f'{save_path}results_{epochs}.pt')
train_losses, test_losses, toy_models_saved, sae_models_saved, disentangled_models_saved, sae_after, train_loader, mmcs_values, wmmcs_values, similarity_scores, frob_norms, permutation_distances = results
'''

"\nresults = train_sweep_of_sae_on_toy_model(\n    n_hidden, \n    n_features, \n    n_dictionary, \n    l1_lambda, \n    save_path, \n    n_sparsities=n_sparsities, \n    epochs=epochs,\n    log_freq=log_freq,\n    n_samples=n_samples,\n    batch_size=batch_size,\n    limit=limit,\n    temperature=temperature,\n    criterion=criterion\n)\ntorch.save(results, f'{save_path}results_{epochs}.pt')\ntrain_losses, test_losses, toy_models_saved, sae_models_saved, disentangled_models_saved, sae_after, train_loader, mmcs_values, wmmcs_values, similarity_scores, frob_norms, permutation_distances = results\n"

In [30]:
import  pysgmcmc.optimizers.sghmc as sgmhc

sampler = sgmhc.SGHMC(
    params=toy_models_saved[-1].parameters(),
    lr=0.01,                 # Learning rate (step size)
    num_burn_in_steps=300,  # Burn-in period to adapt parameters
    noise=0.0,               # Initial per-parameter noise level
    mdecay=0.05,             # Momentum decay
    scale_grad=n_samples  # Scaling factor - typically dataset size
)

In [36]:
num_samples = 100
samples = []

# Utility to extract all model parameters
def get_parameter_vector(model):
    return torch.cat([p.data.flatten() for p in model.parameters()])

# Sampling loop
for step in tqdm(range(500)):  # Total number of steps
    for batch in train_loader:
        batch = batch.to('cuda')
        # 1. Define closure for the optimizer
        def closure():
            sampler.zero_grad()
            nll = toy_model_loss(toy_models_saved[-1], batch)
            nll.backward()
            return nll
        
        # 2. Take a sampling step
        sampler.step(closure)
        
        # 3. After burn-in, collect samples periodically
        if step > sampler.param_groups[0]['num_burn_in_steps'] and step % 2 == 0:
            samples.append(get_parameter_vector(toy_models_saved[-1]).clone().detach())

# Convert samples to tensors for analysis
parameter_samples = torch.stack(samples)

  0%|          | 0/500 [00:00<?, ?it/s]

In [56]:
def sample_tempered_posterior(model, batch, beta, gamma, n_samples=1000, burn_in=500):
    """
    Sample from the tempered posterior p^β(w|D_n, w*, γ)
    Parameters:
    - beta: Temperature parameter
    - gamma: Localization parameter
    - w*: Reference parameter value
    """
    # Keep a copy of the original parameters (w*)
    w_star = {name: param.clone().detach() for name, param in model.named_parameters()}
    
    # Dataset size
    n = batch.size(0)
    
    # Create the SGHMC sampler
    sampler = sgmhc.SGHMC(
        params=model.parameters(),
        lr=0.01,
        num_burn_in_steps=burn_in,
        noise=0.0,
        mdecay=0.05,
        scale_grad=n
    )
    
    # Function to calculate the tempered posterior
    def tempered_posterior_nll():
        # Standard NLL term (scaled by beta)
        standard_nll = beta * toy_model_loss(model,batch)
        
        # Localization term (distance from w*)
        localization_penalty = 0
        for name, param in model.named_parameters():
            localization_penalty += gamma * torch.sum((param - w_star[name])**2)
        
        return standard_nll + localization_penalty
    
    # Collect samples
    samples = []
    
    # Sampling loop
    for step in range(n_samples + burn_in):
        # Define closure for the optimizer
        def closure():
            sampler.zero_grad()
            loss = tempered_posterior_nll()
            loss.backward()
            return loss
        
        # Take a sampling step
        sampler.step(closure)
        
        # After burn-in, collect samples
        if step >= burn_in:
            # Store function evaluations for the samples
            with torch.no_grad():
                current_nll = toy_model_loss(model, batch)
                samples.append(current_nll.item())
    
    return samples

# 4. Function to compute λ(w*) using the formula in the image
def compute_lambda(batch, model, beta=1.0, gamma=0.1):
    # Get dataset size
    n = batch.size(0)
    
    # Compute L_n(w*) - the NLL at the reference parameters
    with torch.no_grad():
        nll_at_w_star = toy_model_loss(model, batch)
    
    # Sample from tempered posterior and compute E[L_n(w)]
    posterior_samples = sample_tempered_posterior(model,batch, beta, gamma)
    expected_nll = np.mean(posterior_samples)
    
    # Compute λ(w*) according to the formula
    lambda_w_star = (expected_nll - nll_at_w_star) / np.log(n)
    
    return lambda_w_star

# Get test data
test_dataloader = get_dataloader([0.999], n_features, 5000, 5000, limit, temperature)
batch = next(iter(test_dataloader)).to('cuda')

# Compute λ(w*)
lambda_value = compute_lambda(batch, toy_models_saved[-1], gamma=10)
print(f"λ(w*) = {lambda_value}")

probs: [0.16666667 0.16666667 0.16666667 0.16666667 0.16666667 0.16666667]
feature 0: 832
feature 1: 802
feature 2: 845
feature 3: 814
feature 4: 833
feature 5: 874
λ(w*) = -2.0751720057887724e-06


In [5]:
#unnormalized_probs =  + (0.1 * np.arange(n_features - 1, -1, -1))
temperature = 0.0
power = 2.0
unnormalized_probs = np.array([1/(power**(i*temperature)) for i in range(n_features)])
probs = unnormalized_probs / unnormalized_probs.sum()
print(np.round(probs, 3))

[0.167 0.167 0.167 0.167 0.167 0.167]


In [6]:
save_path = f"results/"
n_sparsities = 1
sparsities = get_sparsities(n_sparsities)
for sparsity in tqdm(sparsities): 
    sae_state_dict = torch.load(f"{save_path}sae_s{sparsity}_d{n_dictionary}_h{n_hidden}.pth")
    print(f"{save_path}sae_s{sparsity}_d{n_dictionary}_h{n_hidden}.pth")
        
    sae = SparseAutoencoder(n_hidden, n_dictionary)
    sae.load_state_dict(sae_state_dict)
    W_sae = sae.encoder.weight.data

    toy_model = load_model(sparsity, n_features, n_hidden)
    W_toy = toy_model.W.data.T


    def select_top_k(W, k):
        diag = torch.diag(torch.matmul(W, W.T))
        diag, indices = torch.topk(diag, k=k)
        return W[indices]
       
    W_sae = select_top_k(W_sae, n_features)
    visualize_sae_and_toy_model(W_sae, W_toy, sparsity)

  0%|          | 0/1 [00:00<?, ?it/s]

/scratch-local/bshaffrey.12862092/ipykernel_1086788/1492492564.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sae_state_dict = torch.load(f"{save_path}sae_s{sparsity}_d

results/sae_s0.7_d48_h2.pth


In [7]:
# Create evaluator
evaluator_simult = SAEEvaluator(sae, toy_model)
evaluator_after = SAEEvaluator(sae_after, toy_model)

# Get test data
test_dataloader = get_dataloader(sparsity, n_features, 5000, 5000, limit)

# Evaluate
results_simultaneous_training = evaluator_simult.evaluate(test_dataloader)
results_after_training = evaluator_after.evaluate(test_dataloader)


for key, elem in results_simultaneous_training.items():
    print(key, elem)

for key, elem in results_after_training.items():
    print(key, elem)

AttributeError: 'NoneType' object has no attribute 'to'

In [8]:
disentangled_models_saved = []

for toy_model, sae_model in tqdm(zip(toy_models_saved, sae_models_saved), total=len(toy_models_saved)):
    disentangled_models_saved.append(DisentangledModel(toy_model, sae_model))
    

  0%|          | 0/100 [00:00<?, ?it/s]

In [9]:
train_losses_disentangled_model = []
test_losses_disentangled_model = []
test_loader = get_dataloader(sparsity, n_features, 5000, 5000, limit, temperature)

for disentangled_model in tqdm(disentangled_models_saved):
    total_train_loss = 0
    disentangled_model.eval()
    for batch in train_loader:
        batch = batch.to('cuda')
        loss = toy_model_loss(disentangled_model, batch)
        total_train_loss += loss.item()
    train_losses_disentangled_model.append(total_train_loss / len(train_loader))
    
    total_test_loss = 0
    for batch in test_loader:
        batch = batch.to('cuda')
        loss = toy_model_loss(disentangled_model, batch)
        total_test_loss += loss.item()
    test_losses_disentangled_model.append(total_test_loss / len(test_loader))


probs: [0.16666667 0.16666667 0.16666667 0.16666667 0.16666667 0.16666667]
feature 0: 865
feature 1: 813
feature 2: 822
feature 3: 850
feature 4: 843
feature 5: 807


  0%|          | 0/100 [00:00<?, ?it/s]

In [10]:
#train_losses = []
test_losses = []

for model in tqdm(toy_models_saved):
    '''
    total_train_loss = 0
    toy_model.eval()
    for batch in train_loader:
        batch = batch.to('cuda')
        loss = toy_model_loss(model, batch)
        total_train_loss += loss.item()
    train_losses.append(torch.tensor(total_train_loss / len(train_loader)))
    '''
    
    
    total_test_loss = 0
    for batch in test_loader:
        batch = batch.to('cuda')
        loss = toy_model_loss(model, batch)
        total_test_loss += loss.item()
    test_losses.append(total_test_loss / len(test_loader))

  0%|          | 0/100 [00:00<?, ?it/s]

In [11]:
num_gpus = torch.cuda.device_count()
print(num_gpus)

# Properties of each GPU
for i in range(num_gpus):
   props = torch.cuda.get_device_properties(i)
   print(f"GPU {i}: {props.name}")
   print(f"Memory: {props.total_memory/1e9:.2f} GB")
   print(f"Compute Units: {props.multi_processor_count}")
   print("---")

1
GPU 0: NVIDIA A100-SXM4-40GB
Memory: 42.41 GB
Compute Units: 108
---


In [12]:
def moving_average_padded(curve, window_size):
   smoothed = np.zeros_like(curve)
   for i in range(len(curve)):
       # At start of curve, use smaller window that grows
       if i < window_size//2:
           window = curve[0:i + window_size//2 + 1]
           smoothed[i] = np.mean(window)
       # At end of curve, use smaller window that shrinks
       elif i >= len(curve) - window_size//2:
           window = curve[i - window_size//2:]
           smoothed[i] = np.mean(window)
       # In middle, use full window
       else:
           window = curve[i - window_size//2:i + window_size//2 + 1]
           smoothed[i] = np.mean(window)
   return smoothed

def plot_multiple(plot_data_list, save_filename, nrows=2, ncols=3, **kwargs):
    assert len(plot_data_list) == nrows * ncols

    if len(plot_data_list) > 16:
        print('too many plots to include in one figure')
        return
        
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(15, 10))
    
    # Flatten axes array to iterate easily (works for nrows > 1 or ncols > 1)
    axes = axes.flatten()

    for idx, (plot_data, ax) in enumerate(zip(plot_data_list, axes)):
        plot(plot_data, f"{save_filename}_temp_{idx}", fig_info=(fig, ax), **kwargs)  # Pass axes to `plot`
        ax.set_title(f'Plot {idx+1}')
    
    fig.tight_layout()
    fig.savefig(f'results/{save_filename}.png')

def plot(plot_data, save_filename=None, num_x_points=100, log_freq=1, fig_info=None, begin_primary_colors=0, begin_secondary_colors=0):

    sns.set_style("whitegrid")
    x_axis = np.arange(1, num_x_points + 1, log_freq)
    
    if fig_info is None:
        fig, ax1 = plt.subplots(figsize=(10, 6))  # Create a new figure if no axes are provided
    else:
        fig, ax1 = fig_info

    ax2 = None
    
    # First y-axis for losses
    primary_curves = [d for d in plot_data if 'secondary_axis' not in d or not d['secondary_axis']]
    primary_curves_colors = sns.color_palette("muted")[begin_primary_colors : len(primary_curves) + begin_primary_colors]
    all_values_primary_curves = []
    compute_primary_curves_stats = False
    for item, color in zip(primary_curves, primary_curves_colors):
        values = [value.cpu() if isinstance(value, torch.Tensor) else value for value in item['values']]
        values = moving_average_padded(values, item['window_size'])
        ax1.plot(x_axis, values, label=item['label'], color=color)
        ax1.set_xlabel(item['xlabel'])
        ax1.set_ylabel(item['ylabel'])
        if 'stats' in item:
            all_values_primary_curves.append(values)
            compute_primary_curves_stats = True
    ax1.tick_params(axis='y')

    if compute_primary_curves_stats:
        all_values_primary_curves = np.array(all_values_primary_curves)
        mean = all_values_primary_curves.mean(axis=0)
        std = all_values_primary_curves.std(axis=0)
        
        ax1.plot(x_axis, mean, linestyle="--", label='mean', color='black')
        ax1.fill_between(
            x_axis, mean - std, mean + std, color="gray", alpha=0.3, zorder=2
        )

    if save_filename != None:
        # Create first legend for left y-axis
        lines1, labels1 = ax1.get_legend_handles_labels()
        first_legend = ax1.legend(lines1, labels1, loc='upper left', bbox_to_anchor=(0, 1))
        ax1.add_artist(first_legend)  # Add it to the plot but not in final position

    # Second y-axis for learning coefficient
    secondary_curves = [d for d in plot_data if 'secondary_axis' in d and d['secondary_axis']]
    secondary_curves_colors = sns.color_palette("pastel")[-len(secondary_curves) - begin_secondary_colors : -begin_secondary_colors if begin_secondary_colors > 0 else None]
    if secondary_curves:
        ax2 = ax1.twinx()
        for item, color in zip(secondary_curves, secondary_curves_colors):
            values = [value.cpu() if isinstance(value, torch.Tensor) else value for value in item['values']]
            values = moving_average_padded(values, item['window_size'])
            ax2.plot(x_axis, values, label=item['label'], color=color)
            ax2.set_ylabel(item['ylabel'])
        ax2.tick_params(axis='y')
        
        if save_filename != None:
            # Create second legend for right y-axis
            lines2, labels2 = ax2.get_legend_handles_labels()
            ax2.legend(lines2, labels2, loc='upper left', bbox_to_anchor=(0.8, 0.85))
    
        
    if save_filename != None:
        fig.tight_layout()
        fig.savefig(f'results/{save_filename}.png')
    
    return fig, ax1, ax2

plot_data_loss_and_llc = [
    {'values': train_losses, 'xlabel': 'Epoch', 'ylabel': 'Loss', 'label': 'Train Loss Toy', 'window_size': 1},
    {'values': train_losses_disentangled_model, 'xlabel': 'Epoch', 'ylabel': 'Loss', 'label': 'Train Loss Disentangled', 'window_size': 1},
    #{'values': test_losses, 'xlabel': 'Epoch', 'ylabel': 'Loss', 'label': 'Test Loss Toy', 'window_size': 1},
    #{'values': test_losses_disentangled_model, 'xlabel': 'Epoch', 'ylabel': 'Loss', 'label': 'Test Loss Disentangled', 'window_size': 1},
    #{'values': rlct_estimate_toy_full, 'xlabel': 'Epoch', 'ylabel': 'LLC', 'label': 'λ Toy', 'secondary_axis': True, 'window_size': 10},
    #{'values': rlct_estimate_disentangled_full, 'xlabel': 'Epoch', 'ylabel': 'LLC', 'label': 'λ Disentangled', 'secondary_axis': True, 'window_size': 10}
]

_, _, _ = plot(plot_data_loss_and_llc, f'train_test_loss_and_llc_for_toy_and_disentangled_{epochs}', num_x_points=epochs, log_freq=log_freq)

plot_data_sim = [
    {'values': similarity_scores, 'xlabel': 'Epoch', 'ylabel': 'Similarity', 'label': 'Similarity scores', 'window_size': 1}
]

_, _, _ = plot(plot_data_sim, f'sim_scores_{epochs}', num_x_points=epochs, log_freq=log_freq)

#plot(test_losses, 'Epoch', 'Loss', 'Test Loss', f'test_loss_{epochs}', log_freq)
#plot(mmcs_values, 'Epoch', 'MMCS', 'Mean-Max CS', f'mmcs_{epochs}', log_freq)
#plot(wmmcs_values, 'Epoch', 'WMMCS', 'Weighted Mean-Max CS', f'wmmcs_{epochs}', log_freq)
#plot(permutation_distances, 'Epoch', 'Permutation Dist', 'Permutation Distance', f'perm_dist_{epochs}', log_freq)
#plot(similarity_scores, 'Epoch', 'Similarity', 'Similarity Scores', f'sim_scores_{epochs}', log_freq)
#plot(frob_norms, 'Epoch', 'Norm', 'Frobenius Norm of Diff', f'frob_norm_diff_{epochs}', log_freq)

In [13]:
def create_ablation_dataset(n_samples, n_features, current_feature_idx):
   x = torch.zeros(n_samples, n_features)
   lengths = torch.rand(n_samples)
   x[torch.arange(n_samples), current_feature_idx] = lengths
   return x

ablation_dataset = []

for feature in range(n_features):
    ablation_dataset.append(create_ablation_dataset(10_000, n_features, feature))

final_disentangled_model = disentangled_models_saved[-1]

performance_matrix = np.zeros((n_features, n_features))

for feature in range(n_features):
    curr_model = copy.deepcopy(final_disentangled_model)
    
    # Create a copy of the current column
    col = curr_model.encoder[0].weight.data[:, feature].clone()
    
    # Zero out entire tensor
    curr_model.encoder[0].weight.data.zero_()
    
    # Restore only the current column
    curr_model.encoder[0].weight.data[:, feature] = col
    
    for current_feature in range(n_features):
        ablation_data = ablation_dataset[current_feature].to('cuda')
        performance_matrix[feature, current_feature] = toy_model_loss(curr_model, ablation_data)
    
print(performance_matrix)        

[[0.00361376 0.05729698 0.0569269  0.05667479 0.05696242 0.04378046]
 [0.0569286  0.00380866 0.0569269  0.05667479 0.05696242 0.04378046]
 [0.0569286  0.05729698 0.00374655 0.05667479 0.05696242 0.04378046]
 [0.0569286  0.05729698 0.0569269  0.00358299 0.05696242 0.04378046]
 [0.0569286  0.05729698 0.0569269  0.05667479 0.00381306 0.04378046]
 [0.0569286  0.05729698 0.0569269  0.05667479 0.05696242 0.04378004]]


In [14]:
A = torch.rand((2, 6))
B = torch.rand((12, 2))

C = B @ A

C[ : , 1 : ] = 0
A[ : , 1 : ] = 0
print(C)

print(B @ A)

for name, _ in final_disentangled_model.named_parameters():
    print(name)

tensor([[0.7031, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4837, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2982, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4817, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2059, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3258, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.8534, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.9242, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7600, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1633, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5789, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.9745, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]])
tensor([[0.7031, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4837, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2982, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4817, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2059, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.32

In [15]:
from torch.utils.data import DataLoader

device = 'cuda'
data_overall_losses = []
ablation_dataloader =  DataLoader(ablation_dataset, batch_size=n_features)

for model in tqdm(disentangled_models_saved):
    total_loss = 0
    for batch in ablation_dataloader:
        batch = batch.to(device)
        with torch.no_grad():
            loss = toy_model_loss(model, batch)
            total_loss = loss.item()
    data_overall_losses.append(total_loss / len(ablation_dataloader))

data_all_losses_per_feature = [
    #{'values': data_overall_losses, 'xlabel': 'Epoch', 'ylabel': 'Loss', 'label': f'Train Loss Disentangled', 'secondary_axis': False, 'window_size': 1},
    #{'values': test_losses_disentangled_model, 'xlabel': 'Epoch', 'ylabel': 'Loss', 'label': 'Test Loss Disentangled', 'window_size': 1}
]

for feature in range(n_features):
    loss_per_feature = []
    for model in tqdm(disentangled_models_saved):
        with torch.no_grad():
            ablation_data = ablation_dataset[feature].to('cuda')
            loss_per_feature.append(toy_model_loss(model, ablation_data).item())
    data_all_losses_per_feature.append({'values':loss_per_feature, 'xlabel': 'Epoch', 'ylabel': 'Loss', 'label': f'Loss on feature {feature}', 'secondary_axis': False, 'window_size': 1})


_, _, _ = plot(data_all_losses_per_feature, f'all_losses_per_feature_training_{epochs}', num_x_points=epochs, log_freq=log_freq)
    

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

In [16]:
from torch.utils.data import DataLoader

device = 'cuda'
data_overall_losses = []
ablation_dataloader =  DataLoader(ablation_dataset, batch_size=n_features)

for model in tqdm(toy_models_saved):
    total_loss = 0
    for batch in ablation_dataloader:
        batch = batch.to(device)
        with torch.no_grad():
            loss = toy_model_loss(model, batch)
            total_loss = loss.item()
    data_overall_losses.append(total_loss / len(ablation_dataloader))

data_all_losses_per_feature = [
    #{'values': data_overall_losses, 'xlabel': 'Epoch', 'ylabel': 'Loss', 'label': f'Train Loss Disentangled', 'secondary_axis': False, 'window_size': 1},
    #{'values': test_losses_disentangled_model, 'xlabel': 'Epoch', 'ylabel': 'Loss', 'label': 'Test Loss Disentangled', 'window_size': 1}
]

for feature in range(n_features):
    loss_per_feature = []
    for model in tqdm(toy_models_saved):
        with torch.no_grad():
            ablation_data = ablation_dataset[feature].to('cuda')
            loss_per_feature.append(toy_model_loss(model, ablation_data).item())
    data_all_losses_per_feature.append({'values': loss_per_feature, 'xlabel': 'Epoch', 'ylabel': 'Loss', 'label': f'Loss on feature {feature}', 'secondary_axis': False, 'window_size': 1})


_, _, _ = plot(data_all_losses_per_feature, f'all_losses_per_feature_training_toy_{epochs}', num_x_points=epochs, log_freq=log_freq)
    

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

In [19]:
optimal_loss_curves_per_neuron = []

for feature in range(n_features):
    loss_curves_per_feature = np.zeros((n_features, epochs))
    for epoch, model in enumerate(disentangled_models_saved):
        curr_model = copy.deepcopy(model)
    
        # Create a copy of the current column
        col = curr_model.encoder[0].weight.data[:, feature].clone()
    
        # Zero out entire tensor
        curr_model.encoder[0].weight.data.zero_()
    
        # Restore only the current column
        curr_model.encoder[0].weight.data[:, feature] = col
    
        for current_feature in range(n_features):
            ablation_data = feature_datasets[current_feature].to('cuda')
            loss_curves_per_feature[current_feature, epoch] = toy_model_loss(curr_model, ablation_data)
    plot_data = []
    for idx in range(n_features):
        plot_data.append({'values': loss_curves_per_feature[idx], 'xlabel': 'Epoch', 'ylabel': 'Loss', 'label': f'Loss on feature {idx} for neuron {feature}', 'secondary_axis': False, 'window_size': 1})
    _, _, _ = plot(plot_data, f'losses_per_feature_for_neuron_{feature}_{epochs}', num_x_points=epochs, log_freq=log_freq)
    _, _, _ = plot([plot_data[feature]], f'losses_optimal_feature_for_neuron_{feature}_{epochs}', num_x_points=epochs, log_freq=log_freq)
    # we store the loss curves for the feature that a neuron is representing
    optimal_loss_curves_per_neuron.append(loss_curves_per_feature[feature])


In [18]:
def partition_by_feature(dataloader, n_features, ensure_sparse=True):
    """
    Partition data from a dataloader into n_features batches based on sparse vectors.
    
    Args:
        dataloader: PyTorch DataLoader containing the data
        n_features: Number of features (dimension of sparse vectors)
        ensure_sparse: If True, filters out vectors that don't have exactly one non-zero element
    
    Returns:
        List of batches, where each batch contains data for one feature index
    """
    # Collect all data from the dataloader
    all_data = []
    all_labels = []
    has_labels = False
    
    for batch in dataloader:
        if isinstance(batch, (list, tuple)) and len(batch) == 2:
            data, labels = batch
            all_data.append(data)
            all_labels.append(labels)
            has_labels = True
        else:
            # Assume batch is just data without labels
            all_data.append(batch)
    
    # Concatenate all batches
    all_data = torch.cat(all_data, dim=0)
    if has_labels:
        all_labels = torch.cat(all_labels, dim=0)
    
    if ensure_sparse:
        # Check if vectors have exactly one non-zero element
        # Count non-zero elements in each row
        non_zero_count = (all_data != 0).sum(dim=1)
        is_sparse = non_zero_count == 1
        all_data = all_data[is_sparse]
        if has_labels:
            all_labels = all_labels[is_sparse]
        
        print(f"Filtered to {len(all_data)} sparse vectors from original data")

    
    # Find the feature index for each sample (argmax for sparse vectors)
    feature_indices = torch.argmax(torch.abs(all_data), dim=1)
    
    # Group samples by feature index
    feature_batches = []
    for feature_idx in range(n_features):
        # Find all samples that have their non-zero element at this feature index
        mask = feature_indices == feature_idx
        feature_data = all_data[mask]
        
        print(f"Feature {feature_idx}: {len(feature_data)} samples")
        
        if len(feature_data) > 0:
            if has_labels:
                feature_labels = all_labels[mask]
                feature_batches.append((feature_data, feature_labels))
            else:
                feature_batches.append(feature_data)
        else:
            # Empty batch for this feature
            if has_labels:
                empty_data = torch.empty(0, all_data.shape[1])
                empty_labels = torch.empty(0, dtype=all_labels.dtype)
                feature_batches.append((empty_data, empty_labels))
            else:
                feature_batches.append(torch.empty(0, all_data.shape[1]))
    
    return feature_batches


def create_feature_dataloaders(dataloader, n_features, batch_size=None, ensure_sparse=True, shuffle=False):
    """
    Create separate DataLoaders for each feature index.
    
    Args:
        dataloader: Original PyTorch DataLoader
        n_features: Number of features
        batch_size: Batch size for new dataloaders (None = single batch per feature)
        ensure_sparse: If True, filters out vectors that don't have exactly one non-zero element
        shuffle: Whether to shuffle data in new dataloaders
    
    Returns:
        List of DataLoaders, one for each feature
    """
    feature_batches = partition_by_feature(dataloader, n_features, ensure_sparse)
    
    feature_dataloaders = []
    for i, feature_data in enumerate(feature_batches):
        if isinstance(feature_data, tuple):
            # Has labels
            data, labels = feature_data
            if len(data) > 0:
                dataset = TensorDataset(data, labels)
                if batch_size is None:
                    # Single batch containing all data for this feature
                    loader_batch_size = len(dataset)
                else:
                    loader_batch_size = min(batch_size, len(dataset))
                
                feature_dataloader = DataLoader(dataset, batch_size=loader_batch_size, shuffle=shuffle)
            else:
                feature_dataloader = None
        else:
            # No labels
            data = feature_data
            if len(data) > 0:
                dataset = TensorDataset(data)
                if batch_size is None:
                    # Single batch containing all data for this feature
                    loader_batch_size = len(dataset)
                else:
                    loader_batch_size = min(batch_size, len(dataset))
                
                feature_dataloader = DataLoader(dataset, batch_size=loader_batch_size, shuffle=shuffle)
            else:
                feature_dataloader = None
        
        feature_dataloaders.append(feature_dataloader)
    
    return feature_dataloaders

'''
 # Method 2: Get feature dataloaders
print("\n=== Method 2: Feature DataLoaders ===")
feature_dataloaders = create_feature_dataloaders(
    train_loader, 
    n_features, 
    batch_size=2000, # use batch sizes bigger than number of instances to get single batch
    shuffle=False
)
'''
feature_datasets = partition_by_feature(train_loader, n_features, ensure_sparse=True)
    
for i, loader in enumerate(feature_datasets):
    print(len(loader))


Filtered to 10000 sparse vectors from original data
Feature 0: 1703 samples
Feature 1: 1672 samples
Feature 2: 1701 samples
Feature 3: 1691 samples
Feature 4: 1637 samples
Feature 5: 1596 samples
1703
1672
1701
1691
1637
1596


In [18]:
interference_dataset = ablation_dataset[0]

print(ablation_dataset[0][0])

for idx in range(1, n_features - 1):
    interference_dataset += ablation_dataset[idx]

print(interference_dataset[0])

interference_dataloader =  DataLoader(interference_dataset, batch_size=len(interference_dataset))

interference_losses_toy = []
interference_losses_disentangled = []

for toy_model, disentangled_model in tqdm(zip(toy_models_saved, disentangled_models_saved), total=epochs):
    total_loss_disentangled = 0
    total_loss_toy = 0
    for batch in interference_dataloader:
        batch = batch.to(device)
        with torch.no_grad():
            toy_loss = toy_model_loss(toy_model, batch)
            total_loss_toy = toy_loss.item()
            disentangled_loss = toy_model_loss(disentangled_model, batch)
            total_loss_disentangled = disentangled_loss.item()
    interference_losses_toy.append(total_loss_toy / len(interference_dataloader))
    interference_losses_disentangled.append(total_loss_disentangled / len(interference_dataloader))

data_interference_losses = [
    {'values': interference_losses_toy, 'xlabel': 'Epoch', 'ylabel': 'Loss', 'label': f'Interference Loss Toy', 'secondary_axis': False, 'window_size': 1},
    {'values': interference_losses_disentangled, 'xlabel': 'Epoch', 'ylabel': 'Loss', 'label': f'Interference Loss Disentangled', 'secondary_axis': False, 'window_size': 1}
]

_, _, _ = plot(data_interference_losses, f'interference_losses_{epochs}', num_x_points=epochs, log_freq=log_freq)

tensor([0.7740, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000])
tensor([0.7740, 0.9971, 0.7950, 0.7302, 0.9262, 0.0000])


  0%|          | 0/100 [00:00<?, ?it/s]

In [20]:
from devinterp.slt.sampler import estimate_learning_coeff_with_summary, LLCEstimator
from devinterp.utils import evaluate_ce
from devinterp.optim.sgld import SGLD
from devinterp.optim.sgnht import SGNHT
from devinterp.slt.wbic import OnlineWBICEstimator
from devinterp.slt.trace import OnlineTraceStatistics
from devinterp.utils import get_init_loss_multi_batch, default_nbeta

def estimate_rlcts(models, param_dict, method_and_args, train_loader, criterion, device, num_draws, seed):
    
    estimates_results = []
    
    for idx, model in enumerate(tqdm(models)):
        num_chains = 5
        llc_estimator = LLCEstimator(
                                        num_chains=num_chains, 
                                        num_draws=num_draws, 
                                        nbeta=default_nbeta(train_loader),
                                        device=device, 
                                        init_loss=get_init_loss_multi_batch(train_loader, num_chains, model, criterion, device)
                                    )
        wbic_estimator = OnlineWBICEstimator(
            num_chains=num_chains, num_draws=num_draws, n=n_samples, device=device
        )
        trace_stats_llc = OnlineTraceStatistics(llc_estimator, attribute='losses', device=device)
        trace_stats_wbic = OnlineTraceStatistics(wbic_estimator, attribute='wbics', device=device)
        
        for method, optimizer_kwargs in method_and_args:
            results = estimate_learning_coeff_with_summary(
                model,
                train_loader,
                evaluate=criterion,
                optimizer_kwargs=optimizer_kwargs,
                sampling_method=SGNHT if method == "sgnht" else SGLD,
                num_chains=num_chains,
                num_draws=num_draws,
                num_burnin_steps=int(0.2 * num_draws),
                num_steps_bw_draws=1,
                device=device,
                #cores=4,
                #gpu_idxs=[0, 1],
                grad_accum_steps=1,
                seed=seed,
                optimize_over_per_model_param=param_dict,
                #callbacks=[llc_estimator, wbic_estimator, trace_stats_llc, trace_stats_wbic]
                callbacks=[wbic_estimator]
            )
            estimates_results.append(results)
    return estimates_results


In [21]:
criterion = toy_model_loss
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
num_draws = 1000
num_checkpoints_hyperparam_search = epochs
indices_epochs_hyperparameters = np.linspace(0, epochs - 1, num_checkpoints_hyperparam_search).round().astype(int)
method_and_args = [("sgld", {"lr": 1e-2, "localization": 10.0, "noise_level": 1.0})]

'''
estimates_results_disentangled_full = estimate_rlcts(np.array(disentangled_models_saved)[indices_epochs_hyperparameters], None, method_and_args, train_loader, criterion, device, num_draws, seed)
torch.save(estimates_results_disentangled_full, f'{save_path}estimates_results_disentangled_full_hyper_param_search_{epochs}.pt')

rlct_estimate_disentangled_full = []

for results in estimates_results_disentangled_full:
    rlct_estimate_disentangled_full.append(results['llc/mean'])
'''

"\nestimates_results_disentangled_full = estimate_rlcts(np.array(disentangled_models_saved)[indices_epochs_hyperparameters], None, method_and_args, train_loader, criterion, device, num_draws, seed)\ntorch.save(estimates_results_disentangled_full, f'{save_path}estimates_results_disentangled_full_hyper_param_search_{epochs}.pt')\n\nrlct_estimate_disentangled_full = []\n\nfor results in estimates_results_disentangled_full:\n    rlct_estimate_disentangled_full.append(results['llc/mean'])\n"

In [ ]:
criterion = toy_model_loss
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
num_draws = 1000
method_and_args = [("sgld", {"lr": 1e-2, "localization": 10.0, "noise_level": 1.0})]

estimates_results_toy_full = estimate_rlcts(np.array(toy_models_saved)[indices_epochs_hyperparameters], None, method_and_args, train_loader, criterion, device, num_draws, seed)
torch.save(estimates_results_toy_full, f'{save_path}estimates_results_toy_full_hyper_param_search_{epochs}.pt')

rlct_estimate_toy_full = []

for results in estimates_results_toy_full:
    rlct_estimate_toy_full.append(results['llc/mean'])

In [4]:
seed = 42
set_seed(seed)
save_path = f"results/"
n_samples = 1000
n_features = 6
n_hidden = 2
n_dictionary = 48
limit = True
epochs = 100
log_freq = epochs // 100 if epochs >= 100 else 1
results = torch.load(f'{save_path}results_100.pt')
train_losses, test_losses, toy_models_saved, sae_models_saved, disentangled_models_saved, sae_after, train_loader, mmcs_values, wmmcs_values, similarity_scores, frob_norms, permutation_distances = results
sae = sae_models_saved[-1]
toy_model = toy_models_saved[-1]
sparsity = 0.999

save_path = f"results_wrllc/"

rlct_estimates = torch.load(f'{save_path}rlct_estimates_{epochs}.pt')
estimates_results_disentangled_full = torch.load(f'{save_path}estimates_results_disentangled_full_{epochs}.pt')
estimates_results_toy_full = torch.load(f'{save_path}estimates_results_toy_full_{epochs}.pt')

print(len(estimates_results_disentangled_full))
print(len(estimates_results_toy_full))

rlct_estimate_disentangled_full = []
rlct_estimate_toy_full = []

for results_disentangled, results_toy in zip(estimates_results_disentangled_full, estimates_results_toy_full):
    rlct_estimate_disentangled_full.append(results_disentangled['llc/mean'])
    rlct_estimate_toy_full.append(results_toy['llc/mean'])

# modify this and correct issue with saving!
rlct_estimate_disentangled_full = torch.load(f'{save_path}rlct_estimate_disentangled_full_{epochs}.pt')
rlct_estimate_toy_full = torch.load(f'{save_path}rlct_estimate_toy_full_{epochs}.pt')

rlct_estimates_neurons = []

for results in rlct_estimates:
    llc_mean_neuron = []
    for epoch in range(epochs):
        llc_mean_neuron.append(results[epoch]['llc/mean'])
    rlct_estimates_neurons.append(llc_mean_neuron)

/scratch-local/bshaffrey.12862092/ipykernel_1086788/4147922002.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  results = torch.load(f'{save_path}results_100.pt')
/scrat

100
100


/scratch-local/bshaffrey.12862092/ipykernel_1086788/4147922002.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  rlct_estimate_disentangled_full = torch.load(f'{save_path

In [21]:
plot_data_llc_toy_and_disentangled = [
    {'values': rlct_estimate_toy_full, 'xlabel': 'Epoch', 'ylabel': 'LLC', 'label': 'λ Toy', 'secondary_axis': True, 'window_size': 10},
    {'values': rlct_estimate_disentangled_full, 'xlabel': 'Epoch', 'ylabel': 'LLC', 'label': 'λ Disentangled', 'secondary_axis': False, 'window_size': 10}
]

_, _, _ = plot(plot_data_llc_toy_and_disentangled, f'llc_disentangled_{epochs}', num_x_points=len(rlct_estimate_disentangled_full))

In [29]:
import math

rlct_composite = [sum(tup) for tup in zip(*rlct_estimates_neurons)]
test = [math.prod(tup) for tup in zip(*rlct_estimates_neurons)]
test2 = [tup for tup in zip(*rlct_estimates_neurons)]

print(test2[0])

print(rlct_composite[ : 5])
print(test[ : 5])

plot_data_llc_composite_and_full = [
    {'values': rlct_composite, 'xlabel': 'Epoch', 'ylabel': 'LLC', 'label': 'λ Composite', 'secondary_axis': True, 'window_size': 10},
    {'values': rlct_estimate_disentangled_full, 'xlabel': 'Epoch', 'ylabel': 'LLC', 'label': 'λ Disentangled', 'secondary_axis': False, 'window_size': 10}
]

_, _, _ = plot(plot_data_llc_composite_and_full, f'llc_composite_and_disentangled_{epochs}', num_x_points=len(rlct_estimate_disentangled_full))

(-0.0015193530125543475, 0.020528730005025864, 0.031523335725069046, 0.02307317964732647, 0.023987729102373123, 0.030656415969133377)
[0.12825003743637353, 0.15178612433373928, 0.1505206972360611, 0.15030612424016, 0.14919931441545486]
[-1.6682875349225214e-11, 2.4111078593274095e-10, 2.2898213535187893e-10, 2.2651885377133443e-10, 2.1705412896538284e-10]


In [ ]:
def compute_rhat(loss_traces):
    # loss_traces shape: (J chains, L draws)
    J, L = loss_traces.shape
    
    # Chain means
    chain_means = np.mean(loss_traces, axis=1)  # x̄ⱼ
    overall_mean = np.mean(chain_means)         # x̄*
    
    # Between-chain variance
    B = (L/(J-1)) * np.sum((chain_means - overall_mean)**2)
    
    # Within-chain variance
    W = np.mean([1/(L-1) * np.sum((chain - chain_mean)**2) 
                 for chain, chain_mean in zip(loss_traces, chain_means)])
    
    # R-hat (no square root)
    R = ((L-1)/L * W + 1/L * B) / W
    
    return R, chain_means

def assess_convergence_for_epoch(loss_traces_for_epoch):
    rhat, chain_means = compute_rhat(loss_traces_for_epoch[ : , 100 : ])
    
    # Additional metrics
    mean_diff = np.max(chain_means) - np.min(chain_means)
    cv = np.std(chain_means) / np.mean(chain_means) * 100
    
    return {
        'rhat': rhat,
        'mean_diff': mean_diff,
        'cv': cv,
        'converged': rhat < 1.1 and cv < 10
    }

def plot_llc_results(estimates_disentangled, estimates_toy, estimator_trace='loss', form='disentangled_full', plot_traces_toy=False):

    cov_disentangled = []
    cov_toy = []
    r_hat_disentangled = []
    r_hat_toy = []
    
    for idx in range(len(estimates_disentangled)):
        stats_disentangled = assess_convergence_for_epoch(estimates_disentangled[idx][f'{estimator_trace}/trace'])
        cov_disentangled.append(stats_disentangled['cv'])
        r_hat_disentangled.append(stats_disentangled['rhat'])

        stats_toy = assess_convergence_for_epoch(estimates_toy[idx][f'{estimator_trace}/trace'])
        cov_toy.append(stats_toy['cv'])
        r_hat_toy.append(stats_toy['rhat'])
    

    plot_data_cov_llc_disentangled_and_toy = [
        {'values': cov_disentangled, 'xlabel': 'Epoch', 'ylabel': 'cov', 'label': 'CoV disentangled', 'secondary_axis': False, 'window_size': 1},
        {'values': cov_toy, 'xlabel': 'Epoch', 'ylabel': 'cov', 'label': 'CoV toy', 'secondary_axis': False, 'window_size': 1}
    ]

    plot(plot_data_cov_llc_disentangled_and_toy, f'cov_{form}_and_toy_{epochs}', num_x_points=num_checkpoints_hyperparam_search)

    plot_data_rhat_llc_disentangled_and_toy = [
        {'values': r_hat_disentangled, 'xlabel': 'Epoch', 'ylabel': 'r-hat', 'label': 'R-hat disentangled', 'secondary_axis': False, 'window_size': 1},
        {'values': r_hat_toy, 'xlabel': 'Epoch', 'ylabel': 'r-hat', 'label': 'R-hat toy', 'secondary_axis': False, 'window_size': 1}
    ]

    plot(plot_data_rhat_llc_disentangled_and_toy, f'rhat_{form}_and_toy_{epochs}', num_x_points=num_checkpoints_hyperparam_search)

    os.makedirs(f'results/{estimator_trace}_traces_{form}', exist_ok=True)
    plot_data_loss_traces_per_epoch = []

    for epoch in tqdm(range(len(estimates_disentangled))):
        plot_data_loss_traces = []

        for idx in range(len(estimates_disentangled[epoch][f'{estimator_trace}/trace'])):
            plot_data_loss_traces.append({'values': estimates_disentangled[epoch][f'{estimator_trace}/trace'][idx], 'xlabel': 'Draw', 'ylabel': f'{estimator_trace}', 'label': f'{estimator_trace} Chain {idx}', 'secondary_axis': False, 'window_size': 1, 'stats': True})

        plot(plot_data_loss_traces, f'{estimator_trace}_traces_{form}/{estimator_trace}_traces_{epoch}_{epochs}', num_x_points=num_draws)
        plot_data_loss_traces_per_epoch.append(plot_data_loss_traces)

    plot_multiple(plot_data_loss_traces_per_epoch, f'{estimator_trace}_traces_{form}/{estimator_trace}_traces_hyperparameter_{num_checkpoints_hyperparam_search}_checks_{epochs}', int(np.sqrt(num_checkpoints_hyperparam_search)), int(np.sqrt(num_checkpoints_hyperparam_search)), num_x_points=num_draws)

    if not plot_traces_toy:
        return
    
    os.makedirs(f'results/{estimator_trace}_traces_toy', exist_ok=True)

    plot_data_loss_traces_per_epoch = []

    for epoch in tqdm(range(len(estimates_toy))):
        plot_data_loss_traces = []

        for idx in range(len(estimates_toy[epoch][f'{estimator_trace}/trace'])):
            plot_data_loss_traces.append({'values': estimates_toy[epoch][f'{estimator_trace}/trace'][idx], 'xlabel': 'Draw', 'ylabel': f'{estimator_trace}', 'label': f'{estimator_trace} Chain {idx}', 'secondary_axis': False, 'window_size': 1, 'stats': True})

        plot(plot_data_loss_traces, f'{estimator_trace}_traces_toy/{estimator_trace}_traces_{epoch}_{epochs}', num_x_points=num_draws)
        plot_data_loss_traces_per_epoch.append(plot_data_loss_traces)

    plot_multiple(plot_data_loss_traces_per_epoch, f'{estimator_trace}_traces_toy/{estimator_trace}_traces_hyperparameter_{num_checkpoints_hyperparam_search}_checks_{epochs}', int(np.sqrt(num_checkpoints_hyperparam_search)), int(np.sqrt(num_checkpoints_hyperparam_search)), num_x_points=num_draws)


plot_llc_results(estimates_results_disentangled_full, estimates_results_toy_full, estimator_trace='wbic', form='disentangled_full', plot_traces_toy=True)

In [ ]:
num_checkpoints_hyperparam_search = 4
rlct_estimates = []
'''
method_and_args_per_neuron = {
    0:[("sgld", {"lr": 1e-3, "localization": 5.0, "noise_level": 1.0})],
    1:[("sgld", {"lr": 1e-3, "localization": 10.0, "noise_level": 1.0})],
    2:[("sgld", {"lr": 1e-3, "localization": 20.0, "noise_level": 1.0})],
    3:[("sgld", {"lr": 1e-3, "localization": 20.0, "noise_level": 1.0})],
    4:[("sgld", {"lr": 1e-3, "localization": 20.0, "noise_level": 1.0})],
    5:[("sgld", {"lr": 1e-3, "localization": 20.0, "noise_level": 1.0})]
}
'''
method_and_args = [("sgld", {"lr": 1e-2, "localization": 50.0, "noise_level": 1.0})]

for col in tqdm(range(n_features)):
    param_dict = {}
    param_dict['encoder.0.weight'] = torch.zeros_like(final_disentangled_model.encoder[0].weight.data, dtype=torch.bool)
    param_dict['encoder.1.weight'] = torch.zeros_like(final_disentangled_model.encoder[1].weight.data, dtype=torch.bool)
    param_dict['decoder.0.weight'] = torch.zeros_like(final_disentangled_model.decoder[0].weight.data, dtype=torch.bool)
    param_dict['decoder.1.weight'] = torch.zeros_like(final_disentangled_model.decoder[1].weight.data, dtype=torch.bool)
    param_dict['decoder.1.bias'] = torch.zeros_like(final_disentangled_model.decoder[1].bias.data, dtype=torch.bool)
    
    # apply wrLLC only to this neuron
    param_dict['encoder.0.weight'][:, col] = True
    
    rlct_estimate = estimate_rlcts(np.array(disentangled_models_saved)[indices_epochs_hyperparameters], param_dict, method_and_args, train_loader, criterion, device, num_draws, seed)
    rlct_estimates.append(rlct_estimate)

if len(indices_epochs_hyperparameters) < epochs:
    torch.save(rlct_estimates, f'{save_path}rlct_estimates_hyperparameter_search_{epochs}.pt') 
else:
    torch.save(rlct_estimates, f'{save_path}rlct_estimates_{epochs}.pt') 

In [31]:
plot_data_wrllc_and_llc = [
    {'values': rlct_estimate_toy_full, 'xlabel': 'Epoch', 'ylabel': 'LLC', 'label': 'λ Toy', 'secondary_axis': True, 'window_size': 1},
    {'values': rlct_estimate_disentangled_full, 'xlabel': 'Epoch', 'ylabel': 'LLC', 'label': 'λ Disentangled', 'secondary_axis': True, 'window_size': 10}
]

rlct_estimates_neurons = []

for idx, rlct_estimate in enumerate(rlct_estimates):
    wrllc_values = []

    for jdx in range(num_checkpoints_hyperparam_search):
        wrllc_values.append(rlct_estimate[jdx]['llc/mean'])
    plot_data_wrllc_and_llc.append({'values': wrllc_values, 'xlabel': 'Epoch', 'ylabel': 'wrLLC', 'label': f'λ Neuron {idx}', 'secondary_axis': False, 'window_size': 10})
    rlct_estimates_neurons.append(wrllc_values)

_, _, _ = plot(plot_data_wrllc_and_llc, f'wrllc_and_llc_{epochs}', num_x_points=num_checkpoints_hyperparam_search)

In [ ]:
for neuron, estimate in tqdm(enumerate(rlct_estimates), total=n_features):
    plot_llc_results(estimate, estimates_results_toy_full, estimator_trace='wbic', form=f'disentangled, neuron {neuron}')

In [ ]:
torch.save(rlct_estimate_disentangled_full, f'{save_path}rlct_estimate_disentangled_full_{epochs}.pt')
torch.save(rlct_estimate_toy_full, f'{save_path}rlct_estimate_toy_full_{epochs}.pt')
torch.save(rlct_estimates, f'{save_path}rlct_estimates_{epochs}.pt')

In [109]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C

import numpy as np

def is_local_min_abs_derivative(derivatives, index, window=1):
    """
    Check if a point is a local minimum of the absolute first derivative.
    
    Args:
        derivatives: numpy array of derivative values
        index: index to check
        window: number of points to check on either side (default=1)
    
    Returns:
        bool: True if point is a local minimum of absolute derivatives
    """
    # Reject first and last points
    if index == 0 or index == len(derivatives) - 1:
        return False
    
    # Get absolute derivatives
    abs_derivatives = np.abs(derivatives)
    
    # Get left and right bounds for window
    left_bound = max(0, index - window)
    right_bound = min(len(derivatives), index + window + 1)
    
    # Get the window of values
    window_values = abs_derivatives[left_bound:right_bound]
    
    # Check if the center point is the minimum in the window
    center_idx = index - left_bound
    return window_values[center_idx] == np.min(window_values)

# Example usage:
def find_plateau_points(derivatives, tolerance, window=1):
    """
    Find points that are:
    1. Below tolerance in absolute value
    2. Local minima of the absolute derivative
    
    Args:
        derivatives: numpy array of derivative values
        tolerance: threshold for considering a point "close to zero"
        window: window size for local minimum check
    
    Returns:
        list of indices that meet both criteria
    """
    plateau_points = []
    
    for i in range(len(derivatives)):
        # Check if absolute derivative is below tolerance
        if abs(derivatives[i]) < tolerance:
            # Check if it's a local minimum of absolute derivatives
            if is_local_min_abs_derivative(derivatives, i, window):
                plateau_points.append(i)
    
    return plateau_points


def d_dt(steps, values):
    slope = np.zeros(len(steps))

    # Compute Slope and Curvature
    for i in range(1, len(steps) - 1):
        dx1 = steps[i+1] - steps[i]
        dx0 = steps[i] - steps[i-1]
        
        dy1 = values[i+1] - values[i]
        dy0 = values[i] - values[i-1]
        
        slope[i] = (dy1 / dx1 + dy0 / dx0) / 2

    slope[0] = slope[1]
    slope[-1] = slope[-2]

    return slope

def d_dlogt(steps, values):
    slope = np.zeros(len(steps))
    log_steps = np.log(np.array(steps) + 1)

    # Compute Slope and Curvature
    for i in range(1, len(steps) - 1):
        dx1 = log_steps[i+1] - log_steps[i]
        dx0 = log_steps[i] - log_steps[i-1]
        
        dy1 = values[i+1] - values[i]
        dy0 = values[i] - values[i-1]
        
        slope[i] = (dy1 / dx1 + dy0 / dx0) / 2

    slope[0] = slope[1]
    slope[-1] = slope[-2]

    return slope

def dlog_dlogt(steps, values):
    slope = np.zeros(len(steps))
    log_steps = np.log(np.array(steps) + 1)
    log_values = np.log(values)

    # Compute Slope and Curvature
    for i in range(1, len(steps) - 1):
        dx1 = log_steps[i+1] - log_steps[i]
        dx0 = log_steps[i] - log_steps[i-1]
        
        dy1 = log_values[i+1] - log_values[i]
        dy0 = log_values[i] - log_values[i-1]
        
        slope[i] = (dy1 / dx1 + dy0 / dx0) / 2

    slope[0] = slope[1]
    slope[-1] = slope[-2]

    return slope

def mark_stages(items, savefile, fig, ax1, ax2):
    
    for item in items:

        # note that a plateau here will also be a zero crossing that is non-trivial in the sense of not just barely changing signs and quickly changing back
        plateaus = find_plateau_points(item['_derivy'], item['tolerance'])
        print(f"Found plateau points at indices: {plateaus}")
    
        ax1.axhline(y=0, color='gray', linestyle=':', linewidth=1)
        #ax2.axhline(y=0, color='black', linestyle=':', linewidth=1)

        for idx, plateau in enumerate(plateaus):
            if idx == 0:
                ax1.axvline(x=plateau + 1, color=item['color'], linestyle='--', label=item['label'], linewidth=1)
            else:
                ax1.axvline(x=plateau + 1, color=item['color'], linestyle='--', linewidth=1)

    # Get existing slope line legends
    lines1, labels1 = ax1.get_legend_handles_labels()
    #lines2, labels2 = ax2.get_legend_handles_labels()
    
    # Create a new legend for plateau lines
    plateau_lines = [line for line, label in zip(lines1, labels1) if 'plateau' in label]
    plateau_labels = [label for label in labels1 if 'plateau' in label]
    
    # Create slope lines legend
    slope_lines = [line for line, label in zip(lines1, labels1) if 'Slope' in label]
    slope_labels = [label for label in labels1 if 'Slope' in label]
    #slope_lines.extend([line for line, label in zip(lines2, labels2) if 'Slope' in label])
    #slope_labels.extend([label for label in labels2 if 'Slope' in label])
    
    # Place the legends in different positions
    ax1.legend(slope_lines, slope_labels, loc='upper left', bbox_to_anchor=(0.0, 1.0))
    ax1.add_artist(ax1.legend(plateau_lines, plateau_labels, loc='upper right', bbox_to_anchor=(1.0, 0.65)))
    
    fig.tight_layout()
    fig.savefig(f'results/{savefile}')

def find_slope(_x, _y):
    
    _steps = _x.reshape((-1, 1))
    kernel = 1.0 * RBF(length_scale=1.0) + WhiteKernel(noise_level=1e-10)

    # Create a Gaussian Process Regressor
    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, normalize_y=True, alpha=1e-10)

    # Fit the Gaussian Process
    gp.fit(_steps, _y)
    _ypred = gp.predict(_steps)
    _derivy = d_dlogt(_x, _ypred)

    return _derivy

window_size = 5
tolerance = 1e-2
_x = np.arange(1, epochs + 1, log_freq)
_y = moving_average_padded(rlct_estimate_disentangled_full, window_size)
_derivy_full = find_slope(_x, _y)
#plot_slope(_x, _derivy, f'results/slope_rlct_final_disentangled_{epochs}.png')
plot_slope_full_and_neuron = [
    {'values': _derivy_full, 'xlabel': 'Epoch', 'ylabel': 'Slope', 'label': 'Slope Full', 'secondary_axis': False, 'window_size': 1},
    ]

mark_stages_data = [
    {'_derivy':_derivy_full, 'tolerance':tolerance, 'color': sns.color_palette()[0], 'label': f'full plateau'}
]

for idx, neuron_rlct_estimate in enumerate(rlct_estimates_neurons):
    #fig, ax = plt.subplots(figsize=(10, 6))
    _y = moving_average_padded(neuron_rlct_estimate, window_size)
    _derivy_neuron = find_slope(_x, _y)
    plot_slope_full_and_neuron.append( {'values': _derivy_neuron, 'xlabel': 'Epoch', 'ylabel': 'Slope', 'label': f'Slope Neuron {idx}', 'secondary_axis': False, 'window_size': 1})
    mark_stages_data.append({'_derivy':_derivy_neuron, 'tolerance':tolerance, 'color': sns.color_palette()[idx + 1], 'label':f'neuron {idx} plateau'})

    # plot slope of neurons and full model and mark all stages
    fig, ax1, ax2 = plot(plot_slope_full_and_neuron)
    mark_stages(mark_stages_data, f'slope_rlct_neuron_{idx}_and_full_{epochs}.png', fig, ax1, ax2)
    plot_slope_full_and_neuron.pop(-1)
    mark_stages_data.pop(-1)

# plot slope of neurons and mark all stages
#fig, ax1, ax2 = plot(plot_slope_full_and_neuron[1 : ], begin_primary_colors=1)
#mark_stages(mark_stages_data, f'slope_rlct_neurons_{epochs}.png', fig, ax1, ax2)

Found plateau points at indices: [1, 17, 19, 26, 31, 77, 90, 94]
Found plateau points at indices: [1, 6, 9, 13, 15, 21, 45, 67, 74, 82, 84, 92, 96]
Found plateau points at indices: [1, 17, 19, 26, 31, 77, 90, 94]
Found plateau points at indices: [1, 16, 24, 48, 67, 82, 94, 98]
Found plateau points at indices: [1, 17, 19, 26, 31, 77, 90, 94]
Found plateau points at indices: [1, 18, 33, 51, 68, 80, 94, 98]
Found plateau points at indices: [1, 17, 19, 26, 31, 77, 90, 94]
Found plateau points at indices: [1, 14, 29, 48, 62, 71, 81, 85, 93, 98]
Found plateau points at indices: [1, 17, 19, 26, 31, 77, 90, 94]
Found plateau points at indices: [1, 31, 59, 74, 85, 90, 98]
Found plateau points at indices: [1, 17, 19, 26, 31, 77, 90, 94]
Found plateau points at indices: [1, 29, 68, 85, 96]


In [110]:
window_size = 5
tolerance = 1e-2
_x = np.arange(1, epochs + 1, log_freq)
_y = moving_average_padded(rlct_estimate_disentangled_full, window_size)
_derivy_full = find_slope(_x, _y)

plot_data_slope = [{'values': _derivy_full, 'xlabel': 'Epoch', 'ylabel': 'Slope', 'label': f'Slope full', 'secondary_axis': False, 'window_size': 1}]
plot_data_loss = [{'values': train_losses, 'xlabel': 'Epoch', 'ylabel': 'Loss', 'label': f'Loss Full', 'secondary_axis': False, 'window_size': 1}]
mark_stages_data = [{'_derivy':_derivy_full, 'tolerance':tolerance, 'color': sns.color_palette()[idx + 1], 'label':f'full plateau'}]

    
fig, ax1, ax2 = plot(plot_data_slope)
mark_stages(mark_stages_data, f'slope_rlct_full_{epochs}.png', fig, ax1, ax2)

fig, ax1, ax2 = plot(plot_data_loss)
mark_stages(mark_stages_data, f'loss_with_stages_full_{epochs}.png', fig, ax1, ax2)

for idx, neuron_rlct_estimate in enumerate(rlct_estimates_neurons):
    window_size = 5
    tolerance = 1e-2
    #fig, ax = plt.subplots(figsize=(10, 6))
    _y = moving_average_padded(neuron_rlct_estimate, window_size)
    _derivy_neuron = find_slope(_x, _y)
    plot_data_slope = [{'values': _derivy_neuron, 'xlabel': 'Epoch', 'ylabel': 'Slope', 'label': f'Slope Neuron {idx}', 'secondary_axis': False, 'window_size': 1}]
    plot_data_loss = [{'values': optimal_loss_curves_per_neuron[idx], 'xlabel': 'Epoch', 'ylabel': 'Loss', 'label': f'Loss Neuron {idx} on Feature {idx}', 'secondary_axis': False, 'window_size': 1}]
    mark_stages_data = [{'_derivy':_derivy_neuron, 'tolerance':tolerance, 'color': sns.color_palette()[idx + 1], 'label':f'neuron {idx} plateau'}]
    
    
    fig, ax1, ax2 = plot(plot_data_slope)
    mark_stages(mark_stages_data, f'slope_rlct_neuron_{idx}_{epochs}.png', fig, ax1, ax2)
    
    fig, ax1, ax2 = plot(plot_data_loss)
    mark_stages(mark_stages_data, f'loss_with_stages_neuron_{idx}_{epochs}.png', fig, ax1, ax2)


Found plateau points at indices: [1, 17, 19, 26, 31, 77, 90, 94]
Found plateau points at indices: [1, 17, 19, 26, 31, 77, 90, 94]
Found plateau points at indices: [1, 6, 9, 13, 15, 21, 45, 67, 74, 82, 84, 92, 96]
Found plateau points at indices: [1, 6, 9, 13, 15, 21, 45, 67, 74, 82, 84, 92, 96]
Found plateau points at indices: [1, 16, 24, 48, 67, 82, 94, 98]
Found plateau points at indices: [1, 16, 24, 48, 67, 82, 94, 98]
Found plateau points at indices: [1, 18, 33, 51, 68, 80, 94, 98]
Found plateau points at indices: [1, 18, 33, 51, 68, 80, 94, 98]
Found plateau points at indices: [1, 14, 29, 48, 62, 71, 81, 85, 93, 98]
Found plateau points at indices: [1, 14, 29, 48, 62, 71, 81, 85, 93, 98]
Found plateau points at indices: [1, 31, 59, 74, 85, 90, 98]
Found plateau points at indices: [1, 31, 59, 74, 85, 90, 98]
Found plateau points at indices: [1, 29, 68, 85, 96]
Found plateau points at indices: [1, 29, 68, 85, 96]
